# Build Chroma database from cleaned AbbVie social media data

In [59]:
import pandas as pd
import chromadb
from chromadb.config import Settings
import os
from datetime import datetime

In [60]:
CLEANED_DATA_FILE = 'AbbVie_AI_Data_Collection_Cleaned.xlsx' 
CHROMA_DB_PATH = 'chroma_db'

## Functions

In [61]:
def load_data():
    xls = pd.ExcelFile(CLEANED_DATA_FILE)
    df_pr = pd.read_excel(xls, 'Press releases')
    df_tw = pd.read_excel(xls, 'Twitter')

    print(f"Loaded {len(df_pr)} press release records and {len(df_tw)} Twitter records.")

    return df_pr, df_tw

In [63]:
def initialize_chroma():
    """Initialize the ChromaDB client and collection."""
    
    # Create client
    client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

    # Logic to delete existing collection if it exists
    try:
        client.delete_collection("abbvie_social_media")
    except:
        pass

    # Create new collection
    collection = client.create_collection(
        name="abbvie_social_media",
        metadata={"description": "AbbVie press releases and twitter posts"}
    )
        
    return client, collection

In [48]:
def process_press_releases(df_pr, collection):
    """Process press release data and insert into ChromaDB."""

    documents = []
    metadatas = []
    ids = []

    for idx, row in df_pr.iterrows():
        doc_text = f"Title: {row['title']}\n\n{row['verbatim']}" 
        
        metadata = {
            "content_type": "press_release",
            "title": str(row['title']),
            "date": row['date'].strftime('%Y-%m-%d') if not pd.isna(row['date']) else '',
            'topics': row['topics'] if not pd.isna(row['topics']) else [],
            'audience': str(row['audience']) if not pd.isna(row['audience']) else '',
            'link': str(row['link']) if not pd.isna(row['link']) else ''
        }

        # Create unqiue ID for each document
        doc_id = f"pr_{idx}"

        documents.append(doc_text)
        metadatas.append(metadata)
        ids.append(doc_id)
    # Insert into ChromaDB
    collection.add(
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )

    return len(documents)

In [64]:
def process_tweets(df_tw, collection):
    """Process Twitter data and insert into ChromaDB."""

    documents = []
    metadatas = []
    ids = []

    for idx, row in df_tw.iterrows():
        doc_text = str(row['post copy'])
        
        metadata = {
            "content_type": "tweet",
            "date": row['date'].strftime('%Y-%m-%d') if not pd.isna(row['date']) else '',
            'topics': row['topics'] if not pd.isna(row['topics']) else [],
            'audience': str(row['audience']) if not pd.isna(row['audience']) else '',
            'link': str(row['link']) if not pd.isna(row['link']) else '',
            'likes': int(row['Likes']) if not pd.isna(row['Likes']) else 0,
            'shares': int(row['Shares']) if not pd.isna(row['Shares']) else 0,
            'comments': int(row['Comments']) if not pd.isna(row['Comments']) else 0,
            'total_engagement': int(row['Total Engagments (DO NOT TOUCH)']) if not pd.isna(row['Total Engagments (DO NOT TOUCH)']) else 0
        }

        # Create unqiue ID for each document
        doc_id = f"tw_{idx}"

        documents.append(doc_text)
        metadatas.append(metadata)
        ids.append(doc_id)

    # Insert into ChromaDB
    collection.add(
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )

    return len(documents)

# Build database

In [66]:

df_pr, df_tw = load_data()

Loaded 225 press release records and 1244 Twitter records.


In [67]:
client, collection = initialize_chroma()

In [68]:
pr_count = process_press_releases(df_pr, collection)
tw_count = process_tweets(df_tw, collection)

In [69]:
pr_count, tw_count

(225, 1244)

# Verify DB was created correctly

In [70]:
collection.count()
# 225 press releases + 1244 tweets = 1469

1469

In [71]:
collection.query(
    query_texts=['clinical trial'],
    n_results=5
)

{'ids': [['tw_372', 'tw_909', 'tw_1237', 'tw_419', 'tw_417']],
 'embeddings': None,
 'documents': [['[Video description: On-screen text: “35+ investigational medicines as potential treatment options; 300+ active clinical trials around the world; Developing new technologies and next-generation treatments in...”]',
   'It takes a community to carry out #ClinicalTrials. See how our Clinical Development Operations team shared their appreciation for all those who make the clinical trial journey possible this #CTD2022. https://t.co/TtUhKQRZzJ',
   'Interested in learning more about clinical trial #diversity and #inclusion? Register now for a conversation with Kim Ribeiro, in partnership with @bwise_bwiseusa: https://t.co/BvS3BOv2X3 https://t.co/pBMQkFtGIP',
   'When joining clinical trials, Sheila encourages, “Ask me first. A study showed 96% of Black women won’t be asked to join a clinical trial. Yet, 85% said they would join... There is medical mistrust in the Black community, but leave th